In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/GPT-from-Scratch/notebooks/"
os.chdir(PROJECT_DIR)
print(os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Colab Notebooks/GPT-from-Scratch/notebooks


In [ ]:
import json
import os

In [5]:
sft_data = [
    {
        "instruction": "أعد صياغة الجملة التالية بلغة أبسط.",
        "input": "كان الجو معتدلًا والسماء صافية طوال النهار.",
        "output": "كان الطقس لطيفًا والسماء واضحة طوال اليوم."
    },
    {
        "instruction": "أجب عن السؤال التالي بإجابة قصيرة.",
        "input": "ما عاصمة المملكة العربية السعودية؟",
        "output": "عاصمة المملكة العربية السعودية هي الرياض."
    },
    {
        "instruction": "اكتب فقرة قصيرة من ثلاث جمل عن أهمية القراءة.",
        "input": "",
        "output": "القراءة تنمّي المعرفة وتوسع الخيال. وهي تساعد الإنسان على فهم العالم من حوله. كما أنها تقوي اللغة والتفكير."
    }
]

In [6]:
output_path = "../data/finetune/sft_data.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(sft_data, f, ensure_ascii=False, indent=2)

print("Saved starter SFT data.")

Saved starter SFT data.


In [7]:
paraphrase_pairs = [
    ("كان الطفل سعيدًا لأنه وجد كتابًا جديدًا.", "فرح الطفل لأنه عثر على كتاب جديد."),
    ("ذهبت الأسرة إلى الحديقة في الصباح الباكر.", "خرجت الأسرة إلى الحديقة في وقت مبكر من الصباح."),
    ("القراءة تساعد الإنسان على توسيع معرفته.", "تساعد القراءة الإنسان على زيادة معرفته."),
    ("جلس التلاميذ في الفصل بهدوء.", "بقي الطلاب في الصف بهدوء."),
    ("كانت السماء صافية والجو جميلًا.", "كانت السماء واضحة وكان الطقس جميلًا."),
    ("اجتهدت مريم في دراستها طوال الأسبوع.", "بذلت مريم جهدًا كبيرًا في دراستها خلال الأسبوع."),
    ("ساعد خالد صديقه في حمل الحقيبة.", "عاون خالد صديقه في حمل الحقيبة."),
    ("وصل القطار إلى المحطة في الوقت المحدد.", "جاء القطار إلى المحطة في الموعد المحدد."),
    ("يحتاج النجاح إلى صبر وعمل مستمر.", "النجاح يتطلب صبرًا وجهدًا متواصلًا."),
    ("وقف المعلم أمام الطلاب وشرح الدرس.", "شرح المعلم الدرس وهو واقف أمام الطلاب."),
] * 8  # 80 samples

paraphrase_data = [
    {
        "instruction": "أعد صياغة الجملة التالية بلغة أبسط.",
        "input": src,
        "output": tgt
    }
    for src, tgt in paraphrase_pairs[:80]
]

print("Paraphrase samples:", len(paraphrase_data))

Paraphrase samples: 80


In [8]:
qa_pairs = [
    ("ما عاصمة المملكة العربية السعودية؟", "عاصمة المملكة العربية السعودية هي الرياض."),
    ("ما الكوكب المعروف بالكوكب الأحمر؟", "الكوكب المعروف بالكوكب الأحمر هو المريخ."),
    ("كم يومًا في الأسبوع؟", "عدد أيام الأسبوع سبعة أيام."),
    ("ما فائدة القراءة؟", "القراءة تزيد المعرفة وتنمّي التفكير."),
    ("ما عكس كلمة كبير؟", "عكس كلمة كبير هو صغير."),
    ("في أي فصل تتساقط الأوراق غالبًا؟", "تتساقط الأوراق غالبًا في فصل الخريف."),
    ("ما الحيوان الذي يعيش في الماء؟", "السمك من الحيوانات التي تعيش في الماء."),
    ("ما لون السماء في اليوم الصافي؟", "تكون السماء زرقاء في اليوم الصافي."),
    ("ماذا نستخدم للكتابة؟", "نستخدم القلم للكتابة."),
    ("ما أهمية النوم؟", "النوم يساعد الجسم على الراحة واستعادة النشاط."),
] * 7  # 70 samples

qa_data = [
    {
        "instruction": "أجب عن السؤال التالي بإجابة قصيرة.",
        "input": q,
        "output": a
    }
    for q, a in qa_pairs[:70]
]

print("QA samples:", len(qa_data))

QA samples: 70


In [9]:
creative_topics = [
    ("القمر", "القمر يضيء السماء ليلًا. ينظر إليه الناس بإعجاب وهدوء. ويمنح الليل جمالًا خاصًا."),
    ("الحديقة", "الحديقة مكان جميل تمتلئ بالأشجار والزهور. يذهب الناس إليها للراحة واللعب. وهي تبعث في النفس السعادة."),
    ("القراءة", "القراءة نافذة إلى المعرفة. تمنح الإنسان أفكارًا جديدة كل يوم. وهي عادة جميلة ومفيدة."),
    ("الصباح", "يبدأ الصباح بنور هادئ وأمل جديد. يستيقظ الناس ليبدؤوا أعمالهم بنشاط. وفيه شعور جميل بالتجدد."),
    ("المدرسة", "المدرسة مكان للتعلم وبناء المستقبل. يلتقي فيها الطلاب بالمعلمين والأصدقاء. ومنها تبدأ خطوات النجاح."),
    ("البحر", "البحر واسع وجميل المنظر. تجذب أمواجه الناس بهدوئها وصوتها المميز. وهو عالم مليء بالأسرار."),
    ("النجاح", "النجاح ثمرة التعب والاجتهاد. يحتاج إلى صبر واستمرار وعدم يأس. ومن يسع إليه بصدق يصل إليه."),
    ("الطفولة", "الطفولة مرحلة مليئة بالبراءة والفرح. يتعلم فيها الإنسان أولى خطوات الحياة. وتبقى ذكرياتها جميلة في القلب."),
    ("الصديق", "الصديق الحقيقي يقف مع صديقه في الأوقات الصعبة. يشاركه الفرح والحزن بإخلاص. ووجوده يجعل الحياة أجمل."),
    ("المطر", "المطر ينعش الأرض ويمنحها حياة جديدة. ينتظره الناس والزرع بفرح كبير. وبعده تبدو الطبيعة أكثر جمالًا."),
] * 5  # 50 samples

creative_data = [
    {
        "instruction": f"اكتب فقرة قصيرة من ثلاث جمل عن {topic}.",
        "input": "",
        "output": paragraph
    }
    for topic, paragraph in creative_topics[:50]
]

print("Creative samples:", len(creative_data))

Creative samples: 50


In [10]:
sft_data = paraphrase_data + qa_data + creative_data

print("Total SFT samples:", len(sft_data))

output_path = "../data/finetune/sft_data.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(sft_data, f, ensure_ascii=False, indent=2)

print("Final SFT dataset saved.")

Total SFT samples: 200
Final SFT dataset saved.


In [11]:
for i in [0, 79, 80, 149, 150, 199]:
    print(f"Sample {i}:")
    print(sft_data[i])
    print("-" * 50)

Sample 0:
{'instruction': 'أعد صياغة الجملة التالية بلغة أبسط.', 'input': 'كان الطفل سعيدًا لأنه وجد كتابًا جديدًا.', 'output': 'فرح الطفل لأنه عثر على كتاب جديد.'}
--------------------------------------------------
Sample 79:
{'instruction': 'أعد صياغة الجملة التالية بلغة أبسط.', 'input': 'وقف المعلم أمام الطلاب وشرح الدرس.', 'output': 'شرح المعلم الدرس وهو واقف أمام الطلاب.'}
--------------------------------------------------
Sample 80:
{'instruction': 'أجب عن السؤال التالي بإجابة قصيرة.', 'input': 'ما عاصمة المملكة العربية السعودية؟', 'output': 'عاصمة المملكة العربية السعودية هي الرياض.'}
--------------------------------------------------
Sample 149:
{'instruction': 'أجب عن السؤال التالي بإجابة قصيرة.', 'input': 'ما أهمية النوم؟', 'output': 'النوم يساعد الجسم على الراحة واستعادة النشاط.'}
--------------------------------------------------
Sample 150:
{'instruction': 'اكتب فقرة قصيرة من ثلاث جمل عن القمر.', 'input': '', 'output': 'القمر يضيء السماء ليلًا. ينظر إليه الناس بإعجاب وهدوء